# 00.02 — Practicar terminal y rutas en Colab

Este cuaderno (`.ipynb`) acompaña la lección **00.02 — Terminal y rutas** de DS Track.
Practicarás comandos reales de **Bash**: ubicación, rutas, lectura de archivos
y consulta de herramientas. No necesitas instalar Python en tu computadora si
lo ejecutas en Colab: Python y Bash se ejecutan en la máquina del servicio.

## Cómo empezar

1. Descarga este archivo desde el repositorio (en GitHub, usa la descarga del archivo original).
2. Abre [Google Colab](https://colab.research.google.com/), inicia sesión y usa
   **Archivo → Subir cuaderno** (o la opción equivalente de cargar un notebook).
3. Elige un entorno alojado de Python con CPU; no necesitas GPU ni TPU.
4. Lee cada explicación, predice el resultado y pulsa el botón de ejecutar de
   la celda siguiente. Avanza de arriba abajo. No necesitas comprender el código
   Python de preparación: aquí estudiarás los comandos Bash.
5. Para comprobar que es reconstruible, reinicia la sesión y ejecuta todas las
   celdas desde el principio. Descarga tu copia `.ipynb` con tus respuestas y resultados.

Un **notebook** reúne texto y celdas ejecutables. El **kernel** ejecuta sus
celdas; aquí es Python/IPython y `%%bash` le pide ejecutar el resto de una celda
en Bash. `%%bash` debe estar en la primera línea y no se escribe en una terminal normal.

Colab ejecuta en un entorno remoto, aunque navegues desde Windows. Sus rutas y
versiones pertenecen a esa sesión, **no a tu computadora**. Para practicar
PowerShell, utiliza el recorrido de la lección en una terminal de Windows.
También puedes usar este cuaderno en Jupyter con kernel Python/IPython y Bash
disponibles, pero esa alternativa sí requiere preparar un entorno local.

## 1. Preparar una carpeta de práctica

La siguiente celda crea una carpeta temporal nueva, un README, un archivo cuyo
nombre empieza por punto y el mismo CSV sintético de tres ventas de la lección.
También prepara `datos` y `mis apuntes`. No descarga ni clona DS Track, no
instala paquetes y no conecta Google Drive. Esta carpeta **no es el repositorio**.

Cada ejecución de esta celda prepara otra carpeta; no borra la anterior.
El servicio puede eliminar los archivos temporales al finalizar la sesión.
La variable de entorno `DS_TRACK_PRACTICA` conserva la ruta para las celdas Bash.

In [ ]:
# Preparación de la muestra; no necesitas estudiar Python todavía.
import os
import shutil
import tempfile
from pathlib import Path

if shutil.which("bash") is None:
    raise RuntimeError("Se necesita Bash. En Colab elige un entorno alojado de Python.")

practica = Path(tempfile.mkdtemp(prefix="ds_track_0002_"))
(practica / "datos").mkdir()
(practica / "mis apuntes").mkdir()
(practica / "README.md").write_text("# Mi práctica de terminal\n", encoding="utf-8")
(practica / ".nota_oculta").write_text("Ejemplo de nombre con punto inicial.\n", encoding="utf-8")
(practica / "datos" / "ventas_demo.csv").write_text('fecha,producto,unidades,importe_mxn\n2026-01-01,cuaderno,2,80\n2026-01-02,lapiz,5,50\n2026-01-03,carpeta,1,35\n', encoding="utf-8")
os.environ["DS_TRACK_PRACTICA"] = practica.as_posix()
print("Carpeta de práctica:", practica)
print("Muestra preparada: tres ventas; importe total 165 MXN.")

## 2. Ubicación y contenido

`cd` cambia de carpeta, `pwd` muestra la ubicación y `ls` lista contenido.
`ls -la` añade detalles y nombres que empiezan por punto, como `.nota_oculta`.
`$DS_TRACK_PRACTICA` consulta la ruta preparada; las comillas protegen los espacios.
Las líneas con `#` son comentarios.

**Predice:** ¿en cuál de las dos listas aparecerá `.nota_oculta`?
La ruta absoluta puede cambiar; debes reconocer `README.md`, `datos` y `mis apuntes`.

In [ ]:
%%bash
cd "$DS_TRACK_PRACTICA" || exit 1
pwd
ls
ls -la

## 3. Comprobar una ruta y leer un resultado negativo

`test -e` comprueba si existe una ruta; `&&` ejecuta el mensaje solo si la
comprobación tuvo éxito. `if … then … else … fi` permite mostrar ambos casos.
Aquí solo necesitas reconocer esas dos salidas.

**Predice:** `README.md` existe; `no_existe.csv` no existe. Espera un mensaje
afirmativo y después otro que confirme la ausencia. Esa ausencia es intencional.

In [ ]:
%%bash
cd "$DS_TRACK_PRACTICA" || exit 1
test -e README.md && echo "README existe"
if test -e no_existe.csv; then
    echo "El archivo existe"
else
    echo "no_existe.csv no existe: resultado esperado"
fi

## 4. Ruta relativa, carpeta actual y carpeta padre

`datos` es una ruta relativa: parte de donde estás. `.` representa la carpeta
actual y `..` la carpeta padre. En Bash, una ruta absoluta empieza por `/`.

**Predice:** después de `cd datos`, ¿cuál de las rutas localizará el CSV?
Al terminar, `cd ..` debe devolverte a la carpeta de práctica.

In [ ]:
%%bash
cd "$DS_TRACK_PRACTICA" || exit 1
cd datos || exit 1
pwd
test -e ./ventas_demo.csv && echo "CSV encontrado desde datos"
if test -e datos/ventas_demo.csv; then
    echo "Ruta inesperada: revisa la preparación"
else
    echo "datos/ventas_demo.csv no existe desde aquí: ya estás dentro de datos"
fi
cd .. || exit 1
pwd

### Por qué cada celda comienza con `cd`

Cada celda `%%bash` ejecuta un proceso de Bash nuevo: el `cd` de la celda anterior
no conserva su ubicación para la siguiente. Por eso repetimos
`cd "$DS_TRACK_PRACTICA" || exit 1`: entrar a la carpeta o detener esa celda
si no existe. Una terminal interactiva normal sí conserva el directorio entre comandos.
Si reinicias el kernel, ejecuta primero la preparación para recuperar la variable.

## 5. Ruta absoluta y nombres con espacios

`$DS_TRACK_PRACTICA/datos/ventas_demo.csv` combina la ubicación preparada con
el resto de la ruta. Las comillas hacen que `mis apuntes` se lea como un solo nombre.

**Observa:** el CSV se encuentra desde otra carpeta mediante la ruta completa.

In [ ]:
%%bash
cd "$DS_TRACK_PRACTICA" || exit 1
cd "mis apuntes" || exit 1
pwd
test -e "$DS_TRACK_PRACTICA/datos/ventas_demo.csv" && echo "CSV encontrado por ruta completa"

## 6. Inspeccionar un archivo de datos

`*.csv` busca nombres con esa extensión. `head -n 3` muestra las tres primeras
líneas de texto. Un CSV representa una tabla; aquí la primera línea contiene
los nombres de columnas y cada línea siguiente es una venta.

**Resultado esperado:** encabezado `fecha,producto,unidades,importe_mxn`, venta
de cuaderno por 80 MXN y venta de lápices por 50 MXN: dos registros, no tres.
Leer el texto no valida fechas, tipos ni importes. `importe_mxn` ya es el total
de la venta; no lo multipliques por `unidades`.

In [ ]:
%%bash
cd "$DS_TRACK_PRACTICA" || exit 1
ls datos/*.csv
head -n 3 datos/ventas_demo.csv

## 7. Qué herramientas tiene esta sesión

`command -v` consulta qué ejecutable resolverá Bash. `--version` pide su versión.
El bloque repite la consulta para Python, uv y Git; la variable `herramienta`
toma un nombre cada vez. No necesitas aprender los bucles todavía.

**Observa:** si falta una herramienta, se informa sin instalarla. Sus versiones
y rutas son las de este entorno. Esto no equivale a ejecutar el verificador del
proyecto ni comprueba la instalación personal de Windows, Linux o macOS.

In [ ]:
%%bash
for herramienta in python uv git; do
    if command -v "$herramienta"; then
        "$herramienta" --version
    else
        echo "$herramienta: no disponible en esta sesión"
    fi
done

## 8. Ampliación opcional: contar, filtrar y resumir

`wc -l` cuenta saltos de línea, `grep` busca texto y `awk` permite trabajar con
campos. `-F','` indica el separador, `NR > 1` omite el encabezado, `$2` representa
el producto y `$4` el importe. `<` entrega el archivo como entrada al comando.

**Predice leyendo la muestra:** cuatro líneas (encabezado y tres ventas), una
venta de lápices, dos ventas de al menos 50 MXN y suma total `80 + 50 + 35 = 165`.

Estos comandos funcionan para esta muestra sencilla. Separar por comas no es
un lector general de CSV: las comillas pueden encerrar comas o saltos de línea.
`wc -l` tampoco cuenta registros en general y depende del salto final del archivo.

In [ ]:
%%bash
cd "$DS_TRACK_PRACTICA" || exit 1
wc -l < datos/ventas_demo.csv
grep ',lapiz,' datos/ventas_demo.csv
awk -F',' 'NR > 1 && $4 >= 50 { print $2, $4 }' datos/ventas_demo.csv
awk -F',' 'NR > 1 { total += $4 } END { print total }' datos/ventas_demo.csv

## Tu práctica y evidencia

Edita esta celda y responde con tus palabras:

- Mi entorno de ejecución es…; ¿es mi computadora o una máquina remota?
- La carpeta de práctica es… y se diferencia de la copia de DS Track porque…
- Desde `datos`, la ruta relativa al CSV es…; desde su carpeta padre es…
- Una ruta absoluta que comprobé es…
- Al leer tres líneas vi… registros, porque…
- Una herramienta detectada o ausente fue…; eso describe…
- Si cierro la sesión y desaparece la carpeta, puedo reconstruirla mediante…

**Reto:** cambia `head -n 3` por `head -n 2`, predice y explica la diferencia.
Después vuelve a ejecutar todo desde una sesión nueva y contrasta resultados.
Guarda tus explicaciones y descarga tu copia del notebook.

En el lab 00.02 puedes usar este cuaderno como ensayo de ubicación y lectura.
La comprobación del repositorio real y del entorno personal queda pendiente
hasta que dispongas de esa copia y su terminal. No ejecutes el verificador en
esta carpeta de ejemplo ni instales uv solo para obtener una salida positiva.

## Glosario breve

| Término | Significado aquí |
| --- | --- |
| Bash | Shell que interpreta los comandos de estas celdas |
| Directorio actual | Carpeta desde la que se resuelven las rutas relativas |
| Ruta relativa | Ubicación expresada a partir del directorio actual |
| Ruta absoluta | Ubicación completa desde la raíz del sistema de archivos |
| Notebook | Documento con texto y celdas ejecutables |
| Kernel | Proceso que ejecuta las celdas; aquí Python/IPython |
| Entorno remoto | Máquina del servicio que ejecuta el código |
| CSV | Archivo de texto que representa registros y columnas |

## Continuar y consultar

Vuelve a **00.02 — Terminal, rutas y verificación del entorno** y a su **lab 00.02**
en tu copia del curso. El siguiente tema es **00.03 — Git e historial**.

- [Colab: funcionamiento, archivos y límites](https://research.google.com/colaboratory/intl/es/faq.html).
- [IPython: celdas Bash](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-bash).